# Análise da sazonalidade do Tempo de Estadia (TE) – soja exportada, ANTAQ 2019–2025

Gera as análises descritivas de sazonalidade usadas no artigo. Lê o `dataset_modelo.parquet` (gerado pelo
`CreateDataset_v2.ipynb`) e, para a análise de chuva, a tabela bruta de paralisações da ANTAQ.

O que se procura separar:
1. **Variação mensal** – o TE muda entre os meses de chegada?
2. **Recorrência entre anos** – o perfil mensal se repete ano a ano, ou é efeito de um ano específico?
3. **Calendário agrícola** – o TE acompanha o volume de soja embarcado?
4. **Registros de chuva** – as paralisações por chuva variam entre os meses? (registro operacional da ANTAQ,
   não é dado meteorológico: não mede precipitação)
5. **Outras explicações** – o padrão se mantém dentro de cada complexo portuário ou é efeito da mistura de portos?

Todos os resultados são **associações estatísticas**, não relações de causa e efeito.

Unidade do TE: **horas** (a mesma do target do modelo). Dias aparecem só como apoio à leitura.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

BASE = Path.cwd()
FIG = BASE / "figuras"
FIG.mkdir(exist_ok=True)
MESES = ["jan", "fev", "mar", "abr", "mai", "jun", "jul", "ago", "set", "out", "nov", "dez"]
plt.rcParams.update({"font.family": "serif", "font.serif": ["Times New Roman"], "font.size": 10})

df = pd.read_parquet("dataset_modelo.parquet")
print(f"{len(df)} atracações | chegadas de {df['dt_chegada'].min():%d/%m/%Y} a {df['dt_chegada'].max():%d/%m/%Y}")

### 1. Estatística descritiva do TE e de seus componentes

`participacao_mediana` = mediana, entre as atracações, da fração do TE ocupada por cada componente.

In [ ]:
COMP = {"t1_espera_atracacao_h": "T1 – espera para atracação",
        "t2_espera_inicio_op_h": "T2 – espera para início da operação",
        "t3_operacao_h": "T3 – operação",
        "t4_espera_desatracacao_h": "T4 – espera para desatracação",
        "lead_time_h": "TE – tempo de estadia"}

Df_Descritiva = df[list(COMP)].describe(percentiles=[.25, .5, .75, .9]).T
Df_Descritiva["participacao_mediana"] = [(df[c] / df["lead_time_h"]).median() for c in COMP]
Df_Descritiva = Df_Descritiva.rename(index=COMP)
Df_Descritiva.round(2)

### 2. Perfil mensal (todos os anos juntos)

Como 2025 vai só até a chegada de 02/10/2025, os meses de out–dez têm um ano a menos. Por isso o volume é
apresentado como **média por ano** em que o mês aparece.

In [ ]:
anos_por_mes = df.groupby("mes")["ano"].nunique()
Df_Mensal = df.groupby("mes").agg(
    atracacoes=("IDAtracacao", "size"),
    toneladas=("peso_soja_t", "sum"),
    te_mediana_h=("lead_time_h", "median"),
    te_p25_h=("lead_time_h", lambda s: s.quantile(.25)),
    te_p75_h=("lead_time_h", lambda s: s.quantile(.75)),
    t1_mediana_h=("t1_espera_atracacao_h", "median"),
    t3_mediana_h=("t3_operacao_h", "median"),
    fila_mediana=("fila_na_chegada", "median"),
)
Df_Mensal["anos"] = anos_por_mes
Df_Mensal["atracacoes_por_ano"] = Df_Mensal["atracacoes"] / Df_Mensal["anos"]
Df_Mensal["milhoes_t_por_ano"] = Df_Mensal["toneladas"] / Df_Mensal["anos"] / 1e6
Df_Mensal["te_mediana_dias"] = Df_Mensal["te_mediana_h"] / 24
Df_Mensal.index = MESES
Df_Mensal.round(1)

### 3. Recorrência entre anos

- Tabela ano × mês com a mediana do TE (e o número de atracações em cada célula).
- Para cada ano, a correlação de Spearman (ρ) entre o perfil mensal daquele ano e a média dos perfis dos
  **outros** anos. ρ próximo de 1 = o ano repete o padrão dos demais; próximo de 0 = não repete.
- Comparação direta: mediana do TE para chegadas em jan–mar versus abr–jun, ano a ano.

In [ ]:
Df_Ano_Mes = df.pivot_table(index="ano", columns="mes", values="lead_time_h", aggfunc="median")
Df_Ano_Mes_n = df.pivot_table(index="ano", columns="mes", values="lead_time_h", aggfunc="size")
Df_Ano_Mes.columns = Df_Ano_Mes_n.columns = MESES

consistencia = []
for ano in Df_Ano_Mes.index:
    perfil_ano = Df_Ano_Mes.loc[ano].dropna()
    perfil_outros = Df_Ano_Mes.drop(index=ano).mean()[perfil_ano.index]
    rho, p = stats.spearmanr(perfil_ano, perfil_outros)
    d = df[df["ano"] == ano]
    q1 = d.loc[d["mes"].between(1, 3), "lead_time_h"].median()
    q2 = d.loc[d["mes"].between(4, 6), "lead_time_h"].median()
    consistencia.append({"ano": ano, "meses_com_dados": len(perfil_ano), "rho_spearman": rho, "p_valor": p,
                         "mes_maior_te": perfil_ano.idxmax(), "mes_menor_te": perfil_ano.idxmin(),
                         "te_mediana_jan_mar_h": q1, "te_mediana_abr_jun_h": q2, "jan_mar_maior": q1 > q2})
Df_Consistencia = pd.DataFrame(consistencia).set_index("ano")
print(Df_Ano_Mes.round(0))
Df_Consistencia.round(3)

### 4. Teste de Kruskal–Wallis: o TE difere entre os meses?

Teste não paramétrico (o TE tem cauda longa). Com milhares de registros, quase qualquer diferença fica
"significativa"; por isso o tamanho de efeito η²_H é mais informativo: η²_H = (H − k + 1) / (n − k)
(Tomczak; Tomczak, 2014). Leitura usual: 0,01 a 0,06 pequeno, 0,06 a 0,14 moderado, ≥ 0,14 grande.

In [ ]:
def kruskal(dados, grupo):
    grupos = [g["lead_time_h"].to_numpy() for _, g in dados.groupby(grupo)]
    h, p = stats.kruskal(*grupos)
    n, k = len(dados), len(grupos)
    return {"n": n, "grupos": k, "H": h, "p_valor": p, "eta2_H": (h - k + 1) / (n - k)}


Df_Kruskal = pd.DataFrame(
    [{"recorte": "todos os anos (por mês)", **kruskal(df, "mes")},
     {"recorte": "todos os anos (por ano)", **kruskal(df, "ano")}]
    + [{"recorte": f"{ano} (por mês)", **kruskal(df[df["ano"] == ano], "mes")} for ano in sorted(df["ano"].unique())]
).set_index("recorte")
Df_Kruskal.round(4)

### 5. Calendário agrícola e congestionamento

Nível ano-mês (só células com pelo menos 10 atracações): o TE mediano acompanha o volume de soja que chega?
E acompanha a fila de navios de granel sólido no momento da chegada?

In [ ]:
Df_YM = df.groupby(["ano", "mes"]).agg(
    atracacoes=("IDAtracacao", "size"), toneladas=("peso_soja_t", "sum"),
    te_mediana_h=("lead_time_h", "median"), fila_mediana=("fila_na_chegada", "median"))
Df_YM = Df_YM[Df_YM["atracacoes"] >= 10]

rho_vol, p_vol = stats.spearmanr(Df_YM["toneladas"], Df_YM["te_mediana_h"])
rho_fila_ym, p_fila_ym = stats.spearmanr(Df_YM["fila_mediana"], Df_YM["te_mediana_h"])
rho_fila, p_fila = stats.spearmanr(df["fila_na_chegada"], df["lead_time_h"])
rho_fila_t1, p_fila_t1 = stats.spearmanr(df["fila_na_chegada"], df["t1_espera_atracacao_h"], nan_policy="omit")

Df_Correlacoes = pd.DataFrame([
    {"relacao": "toneladas do mês x TE mediano do mês", "nivel": "ano-mês", "n": len(Df_YM), "rho": rho_vol, "p_valor": p_vol},
    {"relacao": "fila mediana do mês x TE mediano do mês", "nivel": "ano-mês", "n": len(Df_YM), "rho": rho_fila_ym, "p_valor": p_fila_ym},
    {"relacao": "fila na chegada x TE", "nivel": "atracação", "n": len(df), "rho": rho_fila, "p_valor": p_fila},
    {"relacao": "fila na chegada x T1", "nivel": "atracação", "n": int(df["t1_espera_atracacao_h"].notna().sum()),
     "rho": rho_fila_t1, "p_valor": p_fila_t1},
]).set_index("relacao")
Df_Correlacoes.round(4)

### 6. Outras explicações: o padrão se mantém dentro de cada complexo?

Se o padrão mensal fosse só efeito da mistura de portos (ex.: portos lentos operando mais em certos meses),
ele não apareceria dentro de cada complexo. Comparamos o perfil mensal de cada um dos principais complexos com o
perfil geral (meses com pelo menos 15 atracações no complexo).

In [ ]:
perfil_geral = df.groupby("mes")["lead_time_h"].median()
principais = df["Complexo Portuário"].value_counts().head(6).index
complexos = []
for complexo in principais:
    d = df[df["Complexo Portuário"] == complexo]
    contagem = d.groupby("mes").size()
    perfil = d.groupby("mes")["lead_time_h"].median()[contagem >= 15]
    rho, p = stats.spearmanr(perfil, perfil_geral[perfil.index])
    complexos.append({"complexo": complexo, "atracacoes": len(d), "meses_validos": len(perfil),
                      "te_mediana_h": d["lead_time_h"].median(), "rho_com_perfil_geral": rho, "p_valor": p,
                      "mes_maior_te": MESES[perfil.idxmax() - 1], "mes_menor_te": MESES[perfil.idxmin() - 1]})
Df_Complexos = pd.DataFrame(complexos).set_index("complexo")
Df_Complexos.round(3)

### 7. Paralisações por chuva registradas durante a estadia

A tabela `TemposAtracacaoParalisacao` registra paralisações ocorridas durante o TE de cada atracação
(Documentação do Estatístico Aquaviário, item 1.2.7). Aqui ela é usada só de forma **descritiva**, para as
próprias atracações de soja: essas paralisações acontecem depois da chegada e por isso **não** entram como
feature do modelo. Registros de chuva não equivalem a medições de precipitação.

Qualidade: durações acima do próprio TE da atracação são inconsistentes e são limitadas ao TE.

In [ ]:
def para_data(serie):
    datas = pd.to_datetime(serie, format="%d/%m/%Y %H:%M:%S", errors="coerce")
    faltando = datas.isna() & serie.notna()
    datas[faltando] = pd.to_datetime(serie[faltando], format="ISO8601", errors="coerce")
    return datas


Df_Par = pd.concat([pd.read_csv(BASE / f"Tabelas_Fato_{ano}" / f"{ano}TemposAtracacaoParalisacao.txt", sep=";",
                                dtype=str, encoding="utf-8-sig") for ano in range(2019, 2026)], ignore_index=True)
Df_Par = Df_Par[Df_Par["IDAtracacao"].isin(df["IDAtracacao"])].copy()
Df_Par["horas"] = ((para_data(Df_Par["DTTermino"]) - para_data(Df_Par["DTInicio"])).dt.total_seconds() / 3600).clip(lower=0)
Df_Par["chuva"] = Df_Par["DescricaoTempoDesconto"].str.contains("Chuva", na=False)

Df_Motivos = (Df_Par.groupby("DescricaoTempoDesconto")
              .agg(registros=("horas", "size"), horas=("horas", "sum"), atracacoes=("IDAtracacao", "nunique"))
              .sort_values("horas", ascending=False))
Df_Motivos["pct_horas"] = 100 * Df_Motivos["horas"] / Df_Motivos["horas"].sum()

chuva = (Df_Par[Df_Par["chuva"]].groupby("IDAtracacao")["horas"].sum()
         .rename("horas_chuva_estadia").reset_index())
Df_Chuva = df[["IDAtracacao", "mes", "lead_time_h", "t3_operacao_h"]].merge(chuva, on="IDAtracacao", how="left")
Df_Chuva["horas_chuva_estadia"] = Df_Chuva["horas_chuva_estadia"].fillna(0)
inconsistentes = int((Df_Chuva["horas_chuva_estadia"] > Df_Chuva["lead_time_h"]).sum())
Df_Chuva["horas_chuva_estadia"] = Df_Chuva[["horas_chuva_estadia", "lead_time_h"]].min(axis=1)
Df_Chuva["teve_chuva"] = Df_Chuva["horas_chuva_estadia"] > 0

Df_Chuva_Mes = Df_Chuva.groupby("mes").agg(
    atracacoes=("IDAtracacao", "size"),
    pct_com_chuva=("teve_chuva", lambda s: 100 * s.mean()),
    horas_chuva_media=("horas_chuva_estadia", "mean"),
    horas_chuva_mediana_quando_ha=("horas_chuva_estadia", lambda s: s[s > 0].median()),
)
Df_Chuva_Mes.index = MESES
rho_chuva_t3, p_chuva_t3 = stats.spearmanr(Df_Chuva["horas_chuva_estadia"], Df_Chuva["t3_operacao_h"], nan_policy="omit")
print(f"Atracações com chuva > TE (limitadas ao TE): {inconsistentes}")
print(f"Spearman horas de chuva x T3 (atracação): rho = {rho_chuva_t3:.3f} (p = {p_chuva_t3:.2g})")
print(Df_Motivos.head(8).round(1))
Df_Chuva_Mes.round(1)

### 8. Figuras para o artigo

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
dados_box = [df.loc[df["mes"] == m, "lead_time_h"] for m in range(1, 13)]
ax[0].boxplot(dados_box, tick_labels=MESES, showfliers=False, medianprops={"color": "black"})
ax[0].set(ylabel="TE (horas)", title="(a) TE por mês de chegada")
ax[1].bar(MESES, Df_Mensal["milhoes_t_por_ano"], color="0.55")
ax[1].set(ylabel="milhões de toneladas por ano", title="(b) Soja embarcada por mês de chegada")
for a in ax:
    a.tick_params(axis="x", labelsize=8)
plt.tight_layout()
plt.savefig(FIG / "fig_te_por_mes.png", dpi=200)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.8))
estilos = ["-o", "-s", "-^", "-v", "-D", "-P", "-X"]
for estilo, (ano, linha) in zip(estilos, Df_Ano_Mes.iterrows()):
    ax.plot(MESES, linha.values, estilo, ms=4, lw=1.2, label=str(ano))
ax.set(ylabel="TE mediano (horas)", xlabel="mês de chegada")
ax.legend(ncol=7, fontsize=8, loc="upper center", bbox_to_anchor=(0.5, 1.13), frameon=False)
plt.tight_layout()
plt.savefig(FIG / "fig_te_ano_mes.png", dpi=200)
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
ax[0].plot(MESES, Df_Mensal["te_mediana_h"], "-o", color="black", ms=4, label="TE")
ax[0].plot(MESES, Df_Mensal["t1_mediana_h"], "--s", color="0.35", ms=4, label="T1 – espera para atracação")
ax[0].plot(MESES, Df_Mensal["t3_mediana_h"], ":^", color="0.5", ms=4, label="T3 – operação")
ax[0].set(ylabel="mediana (horas)", title="(a) TE e componentes por mês")
ax[0].legend(fontsize=8, frameon=False)
ax[1].bar(MESES, Df_Chuva_Mes["pct_com_chuva"], color="0.55")
ax[1].set(ylabel="% das atracações", title="(b) Atracações com paralisação por chuva")
for a in ax:
    a.tick_params(axis="x", labelsize=8)
plt.tight_layout()
plt.savefig(FIG / "fig_componentes_chuva.png", dpi=200)
plt.show()

### 9. Salvando as tabelas

In [ ]:
with pd.ExcelWriter("Resultados_Sazonalidade.xlsx", engine="openpyxl") as writer:
    Df_Descritiva.to_excel(writer, sheet_name="Descritiva")
    Df_Mensal.to_excel(writer, sheet_name="Perfil_Mensal")
    Df_Ano_Mes.to_excel(writer, sheet_name="TE_Ano_Mes")
    Df_Ano_Mes_n.to_excel(writer, sheet_name="N_Ano_Mes")
    Df_Consistencia.to_excel(writer, sheet_name="Consistencia_Anos")
    Df_Kruskal.to_excel(writer, sheet_name="Kruskal_Wallis")
    Df_Correlacoes.to_excel(writer, sheet_name="Correlacoes")
    Df_Complexos.to_excel(writer, sheet_name="Complexos")
    Df_Chuva_Mes.to_excel(writer, sheet_name="Chuva_Mes")
    Df_Motivos.to_excel(writer, sheet_name="Motivos_Paralisacao")